In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [6]:
embeddings.embed_query("Hello AI")

[-0.033388204872608185,
 0.03453969955444336,
 0.05947455018758774,
 0.059286076575517654,
 -0.06353533267974854,
 -0.06819585710763931,
 0.08823321014642715,
 0.0344407856464386,
 -0.03278520330786705,
 -0.01581503450870514,
 0.020981715992093086,
 -0.018340280279517174,
 -0.039832133799791336,
 -0.08047076314687729,
 -0.0144692063331604,
 0.0332648903131485,
 0.014259209856390953,
 -0.034050002694129944,
 -0.142915740609169,
 -0.02308328077197075,
 -0.021380074322223663,
 0.0026335862930864096,
 -0.04729272797703743,
 -0.01075274869799614,
 -0.06866802275180817,
 0.03112509846687317,
 0.07594592869281769,
 0.0011282908963039517,
 0.01163200568407774,
 -0.036039236932992935,
 0.04483766108751297,
 0.018390746787190437,
 0.12672798335552216,
 -0.0013597605284303427,
 0.00820671021938324,
 0.06909970939159393,
 -0.0807635635137558,
 -0.058413147926330566,
 0.053754497319459915,
 0.02622760459780693,
 -0.0068286266177892685,
 -0.05635837838053703,
 0.0032929598819464445,
 -0.072501771152

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
documents=["what is a capital USA?",
           "who is a president of USA?",
           "who is a prime minister of India?"]

In [24]:
my_query="Narendra Modi is prime minister of India?"

In [25]:
document_embedding=embeddings.embed_documents(documents)

In [33]:
len(document_embedding)

3

In [27]:
query_embedding=embeddings.embed_query("my_query")

In [34]:
len(query_embedding)

384

In [29]:
cosine_similarity([query_embedding],document_embedding)

array([[0.05537547, 0.03245074, 0.10331504]])

In [49]:
from sklearn.metrics.pairwise import euclidean_distances
euclidean_distances([query_embedding],document_embedding)

array([[1.37449957, 1.39107821, 1.33916766]])

| Metric            | Similarity Score Range | Behavior                              |
| ----------------- | ---------------------- | ------------------------------------- |
| Cosine Similarity | \[-1, 1]               | Focuses on angle only |
| L2 Distance       | \[0, ∞)                | Focuses on **magnitude + direction**  |


In [21]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

In [59]:
index=faiss.IndexFlatL2(384) #create index for storing data with size of emebedding


In [54]:
index


<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001CF5C355BC0> >

In [61]:
vector_store=FAISS(
    embedding_function=embeddings,# embedding model
    index=index,
    docstore=InMemoryDocstore(),# how data stored ADD an empty dictionary to clear data
    index_to_docstore_id={},# uid with each documents
)

In [62]:
vector_store.add_texts(["AI is future","AI is powerfull","Cats are aggressive"])

['0f626df6-d510-43a7-a11d-320b3a5694d2',
 'cfe51094-631a-408a-8a3f-b3b3f00bcd3a',
 'ca1de03b-e1b0-426f-8c2a-adc27f3239da']

In [55]:
vector_store.index_to_docstore_id

{0: '1425aa96-58b0-4fae-9b01-ce5233d577ab',
 1: '5d2c0a96-6907-4ca1-8845-a7efdbbbaf3f',
 2: 'bfadf1a9-14cc-40fa-902c-033d40aaa5c7'}

In [64]:
result=vector_store.similarity_search("Tell me about AI", k=3)
result

[Document(id='cfe51094-631a-408a-8a3f-b3b3f00bcd3a', metadata={}, page_content='AI is powerfull'),
 Document(id='0f626df6-d510-43a7-a11d-320b3a5694d2', metadata={}, page_content='AI is future'),
 Document(id='ca1de03b-e1b0-426f-8c2a-adc27f3239da', metadata={}, page_content='Cats are aggressive')]

k implies the no of documents to be retrieved based on similarity

| Feature               | `Flat`                | `IVF` (Inverted File Index)        | `HNSW` (Graph-based Index)          |
| --------------------- | --------------------- | ---------------------------------- | ----------------------------------- |
| Type of Search     | Exact                 | Approximate (cluster-based)        | Approximate (graph-based traversal) |
| Speed               | Slow (linear scan)    | Fast (search only in top clusters) | Very Fast (graph walk)              |


| Dataset Size              | Recommended Index                 |
| ------------------------- | --------------------------------- |
| UPTO 1L                     | `IndexFlatL2` or `IndexFlatIP`    |
| UPTO 1M                  | `IndexIVFFlat` or `IndexHNSWFlat` |
| > 1M                      | `IndexIVFPQ` or `IndexHNSWFlat`   |


In [65]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [68]:
index=faiss.IndexFlatIP(384)
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

In [69]:
vector_store.add_documents(documents=documents)

['f03ba72f-535b-4ad0-8590-65a1def7c861',
 'c7476937-7ec1-412a-b034-740e7a65df4c',
 'c1ebf331-546c-4b23-b546-8d308fc0d0b4',
 '7cc2d8dc-ed2e-49f9-9a27-da436c692738',
 '4e94d57b-1ea4-4d89-8e71-fe615b67b50b',
 'aafa01b4-d38f-4281-b821-396985299ac9',
 '1a6f42a8-cf7c-44f7-a670-c2b4a4e60e54',
 'f671110f-aaec-4d2c-96ce-7abff2f0922c',
 'd0ab73ba-23d2-4258-817c-ac8254a0ef5e',
 'ae5b12a6-1a53-4dbe-8cd5-9fd9e0923aaa']

In [73]:
vector_store.similarity_search("Langchain provides abstractions to make working with LLM's easy",
                               #k=5#hyperparameter
                               filter={"source":{"$eq":"tweet"}} #source same as tweet
                               )

[Document(id='c1ebf331-546c-4b23-b546-8d308fc0d0b4', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='f671110f-aaec-4d2c-96ce-7abff2f0922c', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='ae5b12a6-1a53-4dbe-8cd5-9fd9e0923aaa', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='4e94d57b-1ea4-4d89-8e71-fe615b67b50b', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again.")]

In [74]:
result=vector_store.similarity_search("Langchain provides abstractions to make working with LLM's easy",
                                      filter={"source":"news"}
                                      )

In [76]:
result[0].metadata

{'source': 'news'}

In [77]:
result[0].page_content

'Robbers broke into the city bank and stole $1 million in cash.'

retriever

In [81]:
retriever=vector_store.as_retriever(search_kwargs={"k":3})

In [82]:
retriever.invoke("Langchain provides abstractions to make working with LLM's easy")

[Document(id='c1ebf331-546c-4b23-b546-8d308fc0d0b4', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='f671110f-aaec-4d2c-96ce-7abff2f0922c', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='ae5b12a6-1a53-4dbe-8cd5-9fd9e0923aaa', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

In [83]:
vector_store.save_local("today's class is faiss index")#ondisk

In [90]:
new_vector_store=FAISS.load_local(
    "today's class is faiss index",embeddings,allow_dangerous_deserialization=True,
    )

In [92]:
new_vector_store.similarity_search("Langchain")

[Document(id='c1ebf331-546c-4b23-b546-8d308fc0d0b4', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='f671110f-aaec-4d2c-96ce-7abff2f0922c', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='4e94d57b-1ea4-4d89-8e71-fe615b67b50b', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(id='1a6f42a8-cf7c-44f7-a670-c2b4a4e60e54', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

In [13]:
from langchain_community.document_loaders import PyPDFLoader
FILE_PATH=r"D:\agentic_ai_2_0\2-Langchain Basics\2.4-VectorDatabase\FAISS\data\llama2.pdf"

In [14]:
loader=PyPDFLoader(FILE_PATH)

In [97]:
len(loader.load())

77

In [15]:
pages=loader.load()

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

In [18]:
split_docs=splitter.split_documents(pages)

In [19]:
len(split_docs)

615

In [24]:
index=faiss.IndexFlatIP(384)
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [25]:
vector_store.add_documents(documents=split_docs)

['7da4c1bd-9979-478b-887f-1ae73e7ce493',
 '8132166a-793a-4bb5-a083-35f8a923d189',
 '2218ced6-5c98-4f4c-99bb-2705189374a6',
 '7dfded20-b41f-4cea-9ffe-5710b65f485d',
 '78f28827-728b-4eb2-9199-99874c66e6ea',
 'd0895b1c-9dc9-4551-b17c-beeabecaefa7',
 '249b489b-1f9f-4252-a783-4fe490647b04',
 '2b32a46b-7fa9-4cd9-ae78-03cccc42b9f1',
 '7602ab54-ef79-41ff-8e12-a376f0f5c88a',
 '3da050db-1db8-4494-b139-4c98cd7ef4b5',
 '3ed59616-68e8-4837-8428-83e3f6b08406',
 '9ed9fb6c-003b-4ea5-b5e6-ff53643a8316',
 '2d1e3f61-ce91-498d-88c9-091e9eff14e3',
 '079d3d2a-223c-4fdd-b6fc-60fb8c840ca4',
 'cb561524-ec73-4c34-a9cf-0d2e76a358f2',
 '19ea01ff-ae69-4346-9190-7542763ad321',
 'afddbb18-90cb-48c0-ab6d-658d6673376d',
 '7be9393c-410c-421a-b796-b35617f08d48',
 '0d4778f1-2de8-4b51-a0ec-0629bee33b31',
 '92ba51dc-6888-40ed-ba80-3a34bfc0c05a',
 '0fb82d99-1143-42c0-b748-dbae08ce7d98',
 'cb6ee9a3-c078-4bea-87ab-1d7faa0f4fc7',
 '9a478f6f-e8d1-4bd3-a17f-438157e3293e',
 'db179e11-4257-4388-a871-b0c27b3e0d7b',
 'eee1e379-2721-

In [26]:
retriever=vector_store.as_retriever(
    search_kwargs={"k":10}
)

In [27]:
retriever.invoke("what is llama model")

[Document(id='eee1e379-2721-478c-aa6f-e1d8b3271610', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\agentic_ai_2_0\\2-Langchain Basics\\2.4-VectorDatabase\\FAISS\\data\\llama2.pdf', 'total_pages': 77, 'page': 3, 'page_label': '4'}, page_content='work (Section 6), and conclusions (Section 7).\n‡https://ai.meta.com/resources/models-and-libraries/llama/\n§We are delaying the release of the 34B model due to a lack of time to sufficiently red team.\n¶https://ai.meta.com/llama\n‖https://github.com/facebookresearch/llama\n4'),
 Document(id='2383f8be-b275-44e1-843b-9138e8e18e71', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:3

In [28]:
import os
os.environ['GOOGLE_API_KEY']=os.getenv("GOOGLE_API_KEY")

In [29]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model='gemini-2.0-flash')

In [30]:
r=model.invoke("what is eby's dogs name?")
r.content

'Eby, also known as Ebyoung, does not have a dog.'

In [31]:
from langchain import hub
prompt=hub.pull("rlm/rag-prompt")

In [32]:
import pprint #pretty print 

In [33]:
pprint.pprint(prompt.messages)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]

In [34]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough #question on runtime

context(retreiver),prompt(hub),parser(langchain)

In [35]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs) 
#take page content from docs which

In [49]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [51]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [52]:
rag_chain.invoke("whta is llama model?")



'Llama 2 is intended for commercial and research use in English. Tuned models are intended for assistant-like chat, whereas pre-trained models can be adapted for a variety of natural language generation tasks. Llama 2 models outperform Llama 1 models.'